# Virasoro fusion at $c = 0$: Eberle–Flohr and Gaberdiel–Runkel–Wood

Eberle and Flohr (*Virasoro representations and fusion for general augmented minimal models*, J. Phys. A 39 (2006) 15245, arXiv:hep-th/0604097) computed with the Nahm–Gaberdiel–Kausch algorithm many fusion products of the universal Virasoro vertex algebra at $c = c_{2,3} = 0$: products of irreducible modules $\mathcal V(h)$, $h$ in the extended Kac table, giving rank-2 and rank-3 indecomposable modules, with the logarithmic couplings $\beta$ of the rank-2 ones (their Table 2). Gaberdiel, Runkel and Wood (*Fusion rules and boundary conditions in the $c = 0$ triplet model*, J. Phys. A 42 (2009) 325403, arXiv:0905.0916, §2.2) corrected one of their results: $\mathcal V(2) \otimes \mathcal V(2)$ is not the vacuum module $\mathcal V$ but its contragredient $\mathcal V^*$ (and likewise for the products of $\mathcal V(1)$).

This notebook reproduces these computations with `M.fusion_product(N)`, which computes the depth-$d$ quotients $(M \boxtimes N)^{(d)}$: $M \boxtimes N$ modulo the words of modes of total degree $> d$ (a mode $L_{-n}$ has degree $n$). They carry $\Delta(L_0)$ (Jordan blocks: rank), the modes of degree $0, \dots, d$, and the lowering modes $L_n$ as maps to the quotients of depth $d - n$.

In [1]:
import time
import vertex_algebras as va
V = va.Virasoro(QQ, 0, conformal_vector='L'); L = V.gen('L')
_irr = {}


def irr(h):
    """The irreducible module V(h) (the simple quotient of the Verma module)."""
    if h not in _irr:
        _irr[h] = V.verma_module(h).simple_quotient()
    return _irr[h]


def kac(r, s):
    return ((3*r - 2*s)^2 - 1) / 24


print([kac(1, s) for s in range(1, 8)], [kac(2, s) for s in range(1, 8)])

[0, 0, 1/3, 1, 2, 10/3, 5] [5/8, 1/8, -1/24, 1/8, 5/8, 35/24, 21/8]


## The window

The quotients are computed from the tensors $x \otimes y$ of total depth $\le K$ (depth = weight minus the lowest weight of the factor). The result maps onto the true quotient once $K$ reaches far enough, and it only shrinks as $K$ grows. Two things must fit in the window: Nahm's special subspace of the first factor at depth $d$ (which ends below the first null vector of $M$), and the null vectors that generate the maximal submodules of the factors (their relations are what cut the product down; Eberle–Flohr make the same remark about their $\tilde L_{\max}$). Stability over a few consecutive windows is *not* enough: below these levels a spurious state can survive several windows. The generating null vectors are read off the characters: the first level where $\dim \mathcal V(h)_n < p(n)$, and, for the braid-type modules of the bulk of the Kac table, a second level where it drops below $p(n) - p(n - l_1)$:

In [2]:
def generating_null_levels(h, bound=14):
    """The levels of the null vectors generating the maximal submodule of M(h), up to bound."""
    Lh = irr(h)
    dims = [Lh.weight_space(h + n).dimension() for n in range(bound + 1)]
    p = [Partitions(n).cardinality() for n in range(bound + 1)]
    l1 = next((n for n in range(bound + 1) if dims[n] < p[n]), None)
    if l1 is None:
        return []
    l2 = next((n for n in range(l1 + 1, bound + 1) if dims[n] < p[n] - p[n - l1]), None)
    return [l1] if l2 is None else [l1, l2]


def window(h1, h2, d):
    """A window containing the special subspace of V(h1) at depth d and the generating null vectors."""
    g1, g2 = generating_null_levels(h1), generating_null_levels(h2)
    return max([d + 1 + (g1[0] if g1 else 0)] + g1 + g2) + 1


print({h: generating_null_levels(h) for h in (0, 1/3, 5/8, 1/8, -1/24, 2, 1, 5, 7, 10/3, 21/8, 33/8, 35/24)})

{0: [1, 2], 1/3: [3], 5/8: [2], 1/8: [4], -1/24: [6], 2: [3, 5], 1: [4, 6], 5: [7, 10], 7: [5, 8], 10/3: [6], 21/8: [8], 33/8: [4], 35/24: [12]}


## The correction of Gaberdiel–Runkel–Wood

$\mathcal V(0)$ is one-dimensional at $c = 0$ ($L_{-1} \Omega$ and $L_{-2} \Omega$ are null): $\mathcal V(0) \boxtimes \mathcal V(0) = \mathcal V(0)$ and $\mathcal V(0) \boxtimes \mathcal V(h) = 0$ for $h \ne 0$ (GRW (2.8)). Eberle–Flohr found $\mathcal V(2) \boxtimes \mathcal V(2) = \mathcal V$ (their $\mathcal R^{(1)}(0)_2$); with associativity this would give $\mathcal V(0) = \mathcal V(0) \boxtimes \mathcal V = (\mathcal V(0) \boxtimes \mathcal V(2)) \boxtimes \mathcal V(2) = 0$. The depth-0 quotient of $\mathcal V$ is spanned by $\Omega$ alone, but that of $\mathcal V(2) \boxtimes \mathcal V(2)$ is two-dimensional, with $L_0$ conjugate to $\mathrm{diag}(0, 2)$ (GRW (2.16), (2.17)):

In [3]:
print([irr(0).weight_space(n).dimension() for n in range(4)])
print(irr(0).fusion_product(irr(0)).depth_quotient_up_to(0, 3).conformal_weights(),
      irr(0).fusion_product(irr(2)).depth_quotient_up_to(0, 6).dimension())
F = irr(2).fusion_product(irr(2)).depth_quotient_up_to(0, window(2, 2, 0))
print(F.basis())
print(F.l0_matrix(), F.conformal_weights())

[1, 0, 0, 0]
[(0, [1])] 0
[(v, v), (L_{-1} v, v)]
[ 4 -8]
[ 1 -2] [(0, [1]), (2, [1])]


GRW also checked that $L_2$ maps the state of weight 2 to the state of weight 0, as in $\mathcal V^*$ (in $\mathcal V$ itself $L_2 L_{-2} \Omega = \frac c2 \Omega = 0$). Here $L_2$ maps the depth-2 quotient to the depth-0 quotient:

In [4]:
P22 = irr(2).fusion_product(irr(2))
K = window(2, 2, 2)
F2, F0 = P22.depth_quotient_up_to(2, K), P22.depth_quotient_up_to(0, K)
L0 = F2.l0_matrix()
print(F2.conformal_weights())
w2 = [v for v in (L0 - 2).right_kernel().basis() if (F2.projection(0) * v) != 0]   # weight 2, not a raising image
w0 = (L0 - 0).right_kernel().basis()[0]
a, b = F2.mode_matrix(L, 3) * w2[0], F2.projection(0) * w0
print(a != 0, matrix([a, b]).rank() == 1)        # L_2 (weight 2) = nonzero multiple of the weight-0 state

[(0, [1]), (2, [1]), (3, [1]), (4, [1, 1])]
True True


The products of $\mathcal V(1)$, where GRW find $\mathcal V(1) \boxtimes \mathcal V(2) = \mathcal P^*$ and $\mathcal V(1) \boxtimes \mathcal V(1) = \mathcal V^* \oplus \mathcal V(\frac13)$ (instead of Eberle–Flohr's $\mathcal R^{(1)}(0)_1$ and $\mathcal R^{(1)}(0)_2 \oplus \mathcal V(\frac13)$): the depth-0 quotients have the weights $0, 1$ and $0, \frac13, 2$:

In [5]:
for h1, h2 in [(1, 2), (1, 1)]:
    print(h1, h2, irr(h1).fusion_product(irr(h2)).depth_quotient_up_to(0, window(h1, h2, 0)).conformal_weights())

1 2 [(0, [1]), (1, [1])]


1 1 [(0, [1]), (1/3, [1]), (2, [1])]


## Eberle–Flohr's table at depth 0

Their Appendix B, the rows of $\mathcal V(\frac58)$, $\mathcal V(\frac13)$, $\mathcal V(\frac18)$ and $\mathcal V(2)$. At depth 0 a summand shows its generating states: $\mathcal V(h)$ one state of weight $h$; a rank-2 module $\mathcal R^{(2)}(h, h)$ (type A) a Jordan block at $h$; $\mathcal R^{(2)}(h_1, h_2)$, $h_1 < h_2$, the ground state and the logarithmic partner, two weights; $\mathcal R^{(3)}(h_1, h_1, h_2, h_2)$ Jordan blocks of size 2 at $h_1$ and $h_2$. Each product is computed in the window above and compared with the next one:

In [6]:
V_ = lambda h: ('V', (h,)); R2 = lambda a, b: ('R2', (a, b)); R3 = lambda *w: ('R3', w)


def depth0(res):
    """Eberle-Flohr's result -> the depth-0 data [(h, Jordan block sizes)]."""
    out = {}
    for kind, ws in res:
        if kind == 'V':
            out.setdefault(ws[0], []).append(1)
        elif kind == 'R2' and ws[0] == ws[1]:
            out.setdefault(ws[0], []).append(2)
        elif kind == 'R2':
            for w in ws:
                out.setdefault(w, []).append(1)
        else:
            for w in sorted(set(ws)):
                out.setdefault(w, []).append(2)
    return sorted((w, sorted(s, reverse=True)) for w, s in out.items())


table = [
    (5/8, 5/8, [R2(0, 2)]), (5/8, 1/3, [V_(-1/24)]), (5/8, 1/8, [R2(0, 1)]), (5/8, -1/24, [R2(1/3, 1/3)]),
    (5/8, 33/8, [R2(2, 7)]), (5/8, 10/3, [V_(35/24)]), (5/8, 21/8, [R2(1, 5)]), (5/8, 35/24, [R2(1/3, 10/3)]),
    (5/8, 2, [V_(5/8)]), (5/8, 1, [V_(1/8)]), (5/8, 7, [V_(33/8)]), (5/8, 5, [V_(21/8)]),
    (1/3, 1/3, [R2(0, 2), V_(1/3)]), (1/3, 1/8, [R2(1/8, 1/8)]), (1/3, -1/24, [R2(5/8, 5/8), V_(-1/24)]),
    (1/3, 33/8, [V_(35/24)]), (1/3, 10/3, [R2(1, 7), V_(10/3)]), (1/3, 21/8, [R2(5/8, 21/8)]),
    (1/3, 35/24, [R2(1/8, 33/8), V_(35/24)]), (1/3, 2, [V_(1/3)]), (1/3, 1, [R2(0, 1)]), (1/3, 7, [V_(10/3)]),
    (1/3, 5, [R2(2, 5)]),
    (1/8, 1/8, [R2(1/3, 1/3), R2(0, 2)]), (1/8, -1/24, [R3(0, 0, 1, 1)]), (1/8, 33/8, [R2(1, 5)]),
    (1/8, 10/3, [R2(5/8, 21/8)]), (1/8, 21/8, [R2(1/3, 10/3), R2(2, 7)]), (1/8, 2, [V_(1/8)]),
    (1/8, 1, [V_(5/8), V_(-1/24)]), (1/8, 7, [V_(21/8)]), (1/8, 5, [V_(33/8), V_(35/24)]),
    (2, -1/24, [V_(-1/24)]), (2, 33/8, [V_(33/8)]), (2, 10/3, [V_(10/3)]), (2, 21/8, [V_(21/8)]),
    (2, 7, [V_(7)]), (2, 5, [V_(5)]),
]
t0 = time.time()
agree = 0
for h1, h2, res in table:
    P = irr(h1).fusion_product(irr(h2))
    K = window(h1, h2, 0)
    got = P.depth_quotient_up_to(0, K).conformal_weights()
    stable = got == P.depth_quotient_up_to(0, K + 1).conformal_weights()
    ok = got == depth0(res) and stable
    agree += ok
    print('V(%s) x V(%s)  K = %s:  %s%s' % (h1, h2, K, got, '' if ok else '   <- Eberle-Flohr: %s' % depth0(res)))
print('%s of %s agree (%.0f s)' % (agree, len(table), time.time() - t0))

V(5/8) x V(5/8)  K = 4:  [(0, [1]), (2, [1])]
V(5/8) x V(1/3)  K = 4:  [(-1/24, [1])]


V(5/8) x V(1/8)  K = 5:  [(0, [1]), (1, [1])]


V(5/8) x V(-1/24)  K = 7:  [(1/3, [2])]
V(5/8) x V(33/8)  K = 5:  [(2, [1]), (7, [1])]


V(5/8) x V(10/3)  K = 7:  [(35/24, [1])]


V(5/8) x V(21/8)  K = 9:  [(1, [1]), (5, [1])]


V(5/8) x V(35/24)  K = 13:  [(1/3, [1]), (10/3, [1])]
V(5/8) x V(2)  K = 6:  [(5/8, [1])]


... (19 outputs left out)

V(2) x V(10/3)  K = 7:  [(10/3, [1])]


V(2) x V(21/8)  K = 9:  [(21/8, [1])]


V(2) x V(7)  K = 9:  [(7, [1])]


V(2) x V(5)  K = 11:  [(5, [1])]
38 of 38 agree (452 s)


## Rank 2: the logarithmic couplings

A rank-2 module $\mathcal R^{(2)}(h, h + l)$ of types B–F has a ground state $|h\rangle$, the singular vector $\chi = X |h\rangle$ at level $l$ (normalised so that $X$ has the coefficient 1 at $L_{-1}^l$), and a logarithmic partner $\xi$ with $L_0 \xi = (h + l) \xi + \chi$. Eberle–Flohr define $\beta$ by $L_1^l \xi = \beta |h\rangle$ after adding descendants of $|h\rangle$ to $\xi$ so that $L_p \xi = 0$ for $p \ge 2$; the same number, without the gauge, is $X^\dagger \xi = \beta |h\rangle$ ($L_{-n}^\dagger = L_n$; $X^\dagger$ kills the descendants of $|h\rangle$ at level $l$ since $\chi$ is singular). For $\mathcal R^{(2)}(0, 2)$, where $L_{-1}|0\rangle$ is null, they use $X = L_{-2}$. Everything happens in the depth-$l$ quotient: $X$ is a word of modes of degree $\le l$, and $X^\dagger$ maps it to the depth-0 quotient.

In [7]:
def singular_word(h, l):
    """X with X v the singular vector of the Verma module M(h) at level l, normalised at L_{-1}^l: {(n_1, ..., n_k): c} for L_{-n_1} ... L_{-n_k}."""
    Mh = V.verma_module(h)
    B = list(Mh.weight_space(h + l).basis())
    imgs = [{(m, k): c for m in (2, 3) for k, c in L.nproduct(b, m).monomial_coefficients().items()} for b in B]   # L_1, L_2
    cols = sorted({k for r in imgs for k in r}, key=repr)
    x, = matrix(QQ, [[r.get(k, 0) for k in cols] for r in imgs]).left_kernel().basis()
    words = {tuple(-m for (i, m) in b.leading_support()[0]): c for b, c in zip(B, x) if c}
    return {w: c / words[(1,) * l] for w, c in words.items()}


def beta(h1, h2, h, l, K, X=None):
    """The logarithmic coupling of the rank-2 summand R(h, h + l) of V(h1) x V(h2)."""
    P = irr(h1).fusion_product(irr(h2))
    F = P.depth_quotient_up_to(l, K)
    L0 = F.l0_matrix()
    Omega, = (L0 - h).right_kernel().basis()
    X = singular_word(h, l) if X is None else X
    chi = 0
    for w, c in X.items():
        v = Omega
        for n in reversed(w):
            v = F.mode_matrix(L, 1 - n) * v              # L_{-n} = L_(1-n)
        chi += c * v
    xi = (L0 - (h + l)).solve_right(chi)
    out = 0
    for w, c in X.items():                               # (L_{-n_1} ... L_{-n_k})^† = L_{n_k} ... L_{n_1}
        v, G = xi, F
        for n in w:
            v, G = G.mode_matrix(L, n + 1) * v, P.depth_quotient_up_to(G.depth() - n, K)
        out += c * v
    o = F.projection(0) * Omega
    b = out[o.nonzero_positions()[0]] / o[o.nonzero_positions()[0]]
    assert out == b * o
    return b


print(singular_word(5/8, 2), singular_word(0, 1))

{(2,): -3/2, (1, 1): 1} {(1,): 1}


In [8]:
cases = [  # (h1, h2, h, l, window, Eberle-Flohr's beta, X)
    (5/8, 5/8, 0, 2, 6, 5/6, {(2,): 1}),        # R(0, 2)_7
    (1/3, 1/3, 0, 2, 6, -5/8, {(2,): 1}),       # R(0, 2)_5
    (5/8, 1/8, 0, 1, 5, 1/3, None),             # R(0, 1)_5
    (1/3, 1, 0, 1, 7, -1/2, None),              # R(0, 1)_7
    (1/3, 21/8, 5/8, 2, 9, -5, None),           # R(5/8, 21/8)
    (5/8, 35/24, 1/3, 3, 9, 140/27, None),      # R(1/3, 10/3)
    (1/3, 5, 2, 3, 9, -420, None),              # R(2, 5)
]
for h1, h2, h, l, K, expected, X in cases:
    t = time.time()
    b = beta(h1, h2, h, l, K, X)
    print('V(%s) x V(%s): R(%s, %s)  beta = %s  (Eberle-Flohr: %s)  %.0f s' % (h1, h2, h, h + l, b, expected, time.time() - t))

V(5/8) x V(5/8): R(0, 2)  beta = 5/6  (Eberle-Flohr: 5/6)  1 s
V(1/3) x V(1/3): R(0, 2)  beta = -5/8  (Eberle-Flohr: -5/8)  0 s
V(5/8) x V(1/8): R(0, 1)  beta = 1/3  (Eberle-Flohr: 1/3)  0 s


V(1/3) x V(1): R(0, 1)  beta = -1/2  (Eberle-Flohr: -1/2)  0 s


V(1/3) x V(21/8): R(5/8, 21/8)  beta = -5  (Eberle-Flohr: -5)  8 s


V(5/8) x V(35/24): R(1/3, 10/3)  beta = 140/27  (Eberle-Flohr: 140/27)  9 s


V(1/3) x V(5): R(2, 5)  beta = -420  (Eberle-Flohr: -420)  19 s


The two couplings at level 4 need the window 12 and 10–15 minutes each; set `SLOW = True` to recompute them. Computed 2026-10-25: $\mathcal R^{(2)}(1, 5)$ from $\mathcal V(\frac58) \boxtimes \mathcal V(\frac{21}8)$: $\beta = \frac{2800}9$ (603 s); $\mathcal R^{(2)}(\frac18, \frac{33}8)$ from $\mathcal V(\frac13) \boxtimes \mathcal V(\frac{35}{24})$: $\beta = -\frac{700}{81}$ (911 s); both as in Eberle–Flohr's Table 2. Not attempted: the two-parameter modules $\mathcal R^{(2)}(1, 7)$ and $\mathcal R^{(2)}(2, 7)$ (types E, F: $\beta_2$ needs their gauge).

In [9]:
SLOW = False
if SLOW:
    for h1, h2, h, l, K, expected in [(5/8, 21/8, 1, 4, 12, 2800/9), (1/3, 35/24, 1/8, 4, 12, -700/81)]:
        print(h1, h2, beta(h1, h2, h, l, K), expected)

## Rank 3

$\mathcal V(\frac18) \boxtimes \mathcal V(-\frac1{24}) = \mathcal R^{(3)}(0, 0, 1, 1)$ and $\mathcal V(-\frac1{24}) \boxtimes \mathcal V(-\frac1{24}) = \mathcal R^{(3)}(0, 0, 2, 2) \oplus \mathcal R^{(2)}(\frac13, \frac13)$: Jordan blocks of size 3 at weight 1 (depth 1) and weight 2 (depth 2):

In [10]:
for h1, h2, d in [(1/8, -1/24, 1), (-1/24, -1/24, 2)]:
    t = time.time()
    F = irr(h1).fusion_product(irr(h2)).depth_quotient_up_to(d, window(h1, h2, d))
    print('V(%s) x V(%s), depth %s: %s  (%.0f s)' % (h1, h2, d, F.conformal_weights(), time.time() - t))

V(1/8) x V(-1/24), depth 1: [(0, [2]), (1, [3, 1]), (2, [2])]  (0 s)


V(-1/24) x V(-1/24), depth 2: [(0, [2]), (1/3, [2]), (1, [1]), (4/3, [2]), (2, [3, 1, 1]), (7/3, [2, 2]), (3, [2]), (4, [2, 2])]  (34 s)


## Not yet

* Eberle–Flohr's fusions *with* rank-2 modules (giving most of their rank-3 modules): `fusion_product` takes any module, but the staggered modules $\mathcal R^{(2)}$ are not constructed in the library yet (they could be built from their generating states and null vectors, or as fusion products once those are modules).
* The two-parameter couplings of $\mathcal R^{(2)}(1, 7)$ and $\mathcal R^{(2)}(2, 7)$.
* The augmented Yang–Lee model ($c_{2,5}$, their Appendix C): the same code with `va.Virasoro(QQ, -22/5, ...)`.